In [11]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
from google.colab import files
uploaded = files.upload()


Saving Train.csv to Train.csv


In [5]:
df = pd.read_csv("Train.csv")
df.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [6]:
df.shape

(10999, 12)

In [29]:
import pandas as pd
import numpy as np

np.random.seed(21)
n = 1000
df = pd.DataFrame({
    'Product_importance': np.random.choice(['low','medium','high'], n),

    'Delivery_Zone': np.random.choice(
        ['North','South','East','West']*50 +
        [f'Zone_{i}' for i in range(40)],
        n
    ),
    'Feature_A': np.random.normal(0,1,n),
    'Feature_B': np.random.normal(0,1,n)
})

print("Synthetic dataset created")
print("Unique Delivery Zones:", df['Delivery_Zone'].nunique())

df.head()

Synthetic dataset created
Unique Delivery Zones: 44


,Product_importance,Delivery_Zone,Feature_A,Feature_B
0,medium,East,1.240618,-1.018397
1,low,West,-2.090327,-0.328446
2,low,East,0.085844,0.241423
3,low,South,-1.979728,-0.399156
4,low,South,0.959595,-0.978583


In [30]:
print("Unique Importance:", df['Product_importance'].nunique())
print("Unique Delivery Zones:", df['Delivery_Zone'].nunique())

Unique Importance: 3
Unique Delivery Zones: 44


In [31]:
class CategoryReducer(BaseEstimator, TransformerMixin):

    def __init__(self, min_ratio=0.02, replace_label="Other"):
        self.min_ratio = min_ratio
        self.replace_label = replace_label
        self.valid_categories = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        for col in X.columns:
            freq_ratio = X[col].value_counts(normalize=True)
            self.valid_categories[col] = freq_ratio[freq_ratio >= self.min_ratio].index.tolist()
        return self
    def transform(self, X):

        X_new = pd.DataFrame(X).copy()

        for col in self.valid_categories:
            allowed = self.valid_categories[col]

            X_new[col] = X_new[col].apply(
                lambda value: value if value in allowed else self.replace_label
            )
        return X_new

In [32]:
reducer = CategoryReducer(min_ratio=0.02)

df_reduced = reducer.fit_transform(
    df[['Product_importance','Delivery_Zone']]
)
print("Zones BEFORE:", df['Delivery_Zone'].nunique())
print("Zones AFTER :", df_reduced['Delivery_Zone'].nunique())
df_reduced.head()

Zones BEFORE: 44
Zones AFTER : 5


,Product_importance,Delivery_Zone
0,medium,East
1,low,West
2,low,East
3,low,South
4,low,South


In [33]:
class RareLabelEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, min_ratio=0.05, new_tag="Other"):
        self.min_ratio = min_ratio
        self.new_tag = new_tag
        self.common_labels = {}

    def fit(self, X, y=None):

        X = pd.DataFrame(X)

        for column in X.columns:
            freq = X[column].value_counts(normalize=True)
            self.common_labels[column] = freq[freq >= self.min_ratio].index.tolist()

        return self

    def transform(self, X):

        X_new = pd.DataFrame(X).copy()

        for column in self.common_labels:
            X_new[column] = np.where(
                X_new[column].isin(self.common_labels[column]),
                X_new[column],
                self.new_tag
            )
        return X_new

In [35]:
rare_encoder = RareLabelEncoder(min_ratio=0.10)
df_rare = rare_encoder.fit_transform(
    df[['Product_importance','Delivery_Zone']]
)
print("Unique zones before:", df['Delivery_Zone'].nunique())
print("After:", df_rare['Delivery_Zone'].nunique())
df_rare.head()

Unique zones before: 44
After: 5


,Product_importance,Delivery_Zone
0,medium,East
1,low,West
2,low,East
3,low,South
4,low,South


In [36]:
print("Before Encoding:")
print(df['Delivery_Zone'].value_counts().head())
print("\nAfter Encoding:")
print(df_rare['Delivery_Zone'].value_counts().head())

Before Encoding:
Delivery_Zone
South      226
East       214
West       201
North      192
Zone_29     10
Name: count, dtype: int64

After Encoding:
Delivery_Zone
South    226
East     214
West     201
North    192
Other    167
Name: count, dtype: int64
